# Nuvoton Industry Project -- Elevator People Counting on the M55M1

This notebook covers the following steps to calibrate and export the YOLO model onto monochrome device specifications:
1. **Conda environment setup** running inside Colab with `condacolab`.
2. **Dataset downloads and layouts** from HuggingFace to a 90/10 layout split.
3. **Local training running** with constant 192x192 monochrome variables triggers inputs.
4. **Evaluation metrics validation** layout scripts.
5. **Onnx/TFLite layout conversion exports** matching Ethos-U55 parameters.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install_from_url("https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh")

In [ ]:
!conda --version

In [ ]:
GIT_REPO_URL = "https://github.com/bencejdanko/m55m1-ElevatorCounting-YOLOv8n"
!git clone {GIT_REPO_URL}

import os
dir_name = os.path.basename(GIT_REPO_URL).replace('.git', '')
%cd {dir_name}/yolov8_ultralytics

In [ ]:
!conda create --name yolov8_DG python=3.10 -y

!conda run -n yolov8_DG python -m pip install --upgrade pip setuptools >> /dev/null
!conda run -n yolov8_DG python -m pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu118 >> /dev/null

# Install current local package using .[export]
!conda run -n yolov8_DG python -m pip install .[export]

# Additional supporting packages needed for download/export
!conda run -n yolov8_DG python -m pip install datasets onnx2tf ethos-u-vela onnx h5py >> /dev/null

In [ ]:
# Download bdanko/overhead-person-detection dataset
# more info https://github.com/bencejdanko/m55m1-ElevatorCounting-YOLOv8n
!conda run -n yolov8_DG python download_hf_dataset.py --dataset bdanko/overhead-person-detection --out-dir datasets/overhead_monochrome

In [ ]:
# Adjust epochs as needed to avoid overfitting benchmarks
!conda run -n yolov8_DG python dg_train.py \
  --model-cfg relu6-yolov8.yaml \
  --data datasets/overhead_monochrome/data.yaml \
  --imgsz 192 \
  --weights yolov8n.pt \
  --epochs 10 \
  --device 0

In [ ]:
# validation
!conda run -n yolov8_DG python dg_val.py \
  --weights runs/train/exp/weights/best.pt \
  --data datasets/overhead_monochrome/data.yaml \
  --imgsz 192 \
  --device 0

In [ ]:
# graph intermediary to onnx vectors
!conda run -n yolov8_DG python nu_export_tflite_int8.py --format onnx --weights ./runs/train/exp/weights/best.pt --img 192

# calibrations referencing saved monochrome folder
!conda run -n yolov8_DG python generate_calib_data.py \
  --img-size 192 192 \
  --n-img 4 \
  -o calib_data_192_n4_rgb.npy \
  --img-dir datasets/overhead_monochrome/train/images

In [ ]:
# Run onnx2tf to extract TFLite integers graph fully
!rm -rf saved_model
!conda run -n yolov8_DG onnx2tf -i runs/train/exp/weights/best.onnx -oiqt -cind images calib_data_192_n4_rgb.npy "[[[[0,0,0]]]]" "[[[[1,1,1]]]]"